Directional topological descriptor comparison

In [ ]:
import numpy as np
import pandas as pd
import gudhi as gd
from utils import LowerStarLayer, ecg_comparison
from tqdm.auto import tqdm
import wfdb

def load_mitbih_beats(record_id='100', window=200):
    record = wfdb.rdrecord(f'mitbih_data/{record_id}')
    annotation = wfdb.rdann(f'mitbih_data/{record_id}', 'atr')
    signal = record.p_signal[:, 0]
    X = []
    y = []
    for i, label in enumerate(annotation.symbol):
        if label not in ['N', 'V']:
            continue
        idx = annotation.sample[i]
        if idx - window < 0 or idx + window > len(signal):
            continue
        beat = signal[idx - window:idx + window]
        X.append(beat)
        y.append(0 if label == 'N' else 1)
    return (np.array(X), np.array(y))

def load_mitbih_all(records_id, window=200, classes=('N', 'V')):
    """
    Loads all MIT-BIH records into a single dataset.

    Returns:
        X: (n_beats, 2*window)
        y: (n_beats,)
    """
    X_all = []
    y_all = []
    for rec in tqdm(records_id):
        try:
            record = wfdb.rdrecord(f'mitbih_data/{rec}')
            ann = wfdb.rdann(f'mitbih_data/{rec}', 'atr')
            signal = record.p_signal[:, 0]
            for i, label in enumerate(ann.symbol):
                if label not in classes:
                    continue
                idx = ann.sample[i]
                if idx - window < 0 or idx + window >= len(signal):
                    continue
                beat = signal[idx - window:idx + window]
                X_all.append(beat)
                y_all.append(0 if label == classes[0] else 1)
        except Exception as e:
            print(f'Skipping record {rec}: {e}')
    return (np.array(X_all), np.array(y_all))

def balance_dataset(X, y):
    idx_n = np.where(y == 0)[0]
    idx_v = np.where(y == 1)[0]
    idx_n = np.random.choice(idx_n, size=len(idx_v), replace=False)
    idx = np.concatenate([idx_n, idx_v])
    return (X[idx], y[idx])

def build_lowerstar_layer(n_points):
    stbase = gd.SimplexTree()
    for i in range(n_points - 1):
        stbase.insert([i, i + 1], -10000000000.0)
    layer = LowerStarLayer(simplextree=stbase)
    return layer

np.random.seed(42)
records = ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', '112', '113', '114', '115', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '214']
X, y = load_mitbih_all(records_id=records)
print(X.shape)
print(y.shape)
unique, counts = np.unique(y, return_counts=True)
dict(zip(unique, counts))
X, y = balance_dataset(X, y)
print(X.shape)
print(y.shape)
unique, counts = np.unique(y, return_counts=True)
dict(zip(unique, counts))

n_points = 400
layer = build_lowerstar_layer(n_points)
results, timing, folds = ecg_comparison(X, y, layer, n_points, output_prefix="mitbih")

display(results)
display(timing)

100%|██████████| 35/35 [00:03<00:00, 11.62it/s]


(56906, 400)
(56906,)
(9344, 400)
(9344,)


Raw + SVM: 100%|██████████| 50/50 [41:15<00:00, 49.51s/it]


{'Representation': 'Raw', 'Dimension': 400, 'Model': 'SVM', 'Accuracy Mean': np.float64(0.9080906667216608), 'Accuracy Std': np.float64(0.005571196810983159), 'Balanced Accuracy Mean': np.float64(0.9080907945814107), 'Balanced Accuracy Std': np.float64(0.005570382559377053), 'F1 Mean': np.float64(0.9083088551530125), 'F1 Std': np.float64(0.005646495218861671), 'AUC Mean': np.float64(0.9610522348563824), 'AUC Std': np.float64(0.0036673619500581125), 'Fit seconds Mean': np.float64(49.01274850000075), 'Fit seconds Std': np.float64(4.034464875897839), 'Predict seconds Mean': np.float64(0.472188883999479), 'Predict seconds Std': np.float64(0.02832161671181435)}


Raw + Random Forest: 100%|██████████| 50/50 [20:20<00:00, 24.42s/it]


{'Representation': 'Raw', 'Dimension': 400, 'Model': 'Random Forest', 'Accuracy Mean': np.float64(0.9856057299131668), 'Accuracy Std': np.float64(0.0019739913150921776), 'Balanced Accuracy Mean': np.float64(0.9856057094435984), 'Balanced Accuracy Std': np.float64(0.0019739819550747186), 'F1 Mean': np.float64(0.9855344734408001), 'F1 Std': np.float64(0.0020055108266510933), 'AUC Mean': np.float64(0.9988395744728953), 'AUC Std': np.float64(0.00034444276670596107), 'Fit seconds Mean': np.float64(24.285656012000107), 'Fit seconds Std': np.float64(3.3719754941141518), 'Predict seconds Mean': np.float64(0.1150990599990473), 'Predict seconds Std': np.float64(0.030475389927232158)}


... (35 outputs left out)

{'Representation': 'BCT', 'Dimension': 6400, 'Model': 'Random Forest', 'Accuracy Mean': np.float64(0.9764982419115903), 'Accuracy Std': np.float64(0.003387752374401742), 'Balanced Accuracy Mean': np.float64(0.9764983453377458), 'Balanced Accuracy Std': np.float64(0.0033877756204950837), 'F1 Mean': np.float64(0.9766355064142738), 'F1 Std': np.float64(0.003323317622814481), 'AUC Mean': np.float64(0.9979628637604485), 'AUC Std': np.float64(0.00043681425867812584), 'Fit seconds Mean': np.float64(27.435617060001825), 'Fit seconds Std': np.float64(0.3485056970557963), 'Predict seconds Mean': np.float64(0.25648106399807147), 'Predict seconds Std': np.float64(0.022972639405150815)}


BCT + XGBoost: 100%|██████████| 50/50 [15:53<00:00, 19.06s/it]


{'Representation': 'BCT', 'Dimension': 6400, 'Model': 'XGBoost', 'Accuracy Mean': np.float64(0.9820847697643166), 'Accuracy Std': np.float64(0.0027434895257363702), 'Balanced Accuracy Mean': np.float64(0.9820849660479338), 'Balanced Accuracy Std': np.float64(0.0027433890379876037), 'F1 Mean': np.float64(0.982111761481099), 'F1 Std': np.float64(0.002724546029596058), 'AUC Mean': np.float64(0.998489995363959), 'AUC Std': np.float64(0.0003748823321751712), 'Fit seconds Mean': np.float64(18.959761988000245), 'Fit seconds Std': np.float64(0.127656218332392), 'Predict seconds Mean': np.float64(0.02923827800084837), 'Predict seconds Std': np.float64(0.003496355310861533)}


PLT + SVM:  14%|█▍        | 7/50 [4:19:16<26:35:19, 2226.04s/it]